# Advanced Plotly Notes

This notebook is a focused study file for the **Plotly** Python visualization library. It includes proper markdown notes and commented code examples for building interactive charts, dashboards, subplots, maps, animations, and 3D visualizations.

## Learning Goals

- Understand Plotly Express and Graph Objects.
- Build interactive line, bar, scatter, pie, histogram, box, heatmap, map, and 3D plots.
- Customize hover labels, colors, templates, legends, axes, and layouts.
- Create subplots and animation-style visualizations.
- Export charts as HTML for sharing.

> Note: Plotly charts are interactive. In Jupyter, you can hover, zoom, pan, select, and save chart images from the chart toolbar.

## 1. Import and Setup

Plotly has two main APIs:

- `plotly.express`: high-level API for quick, clean charts.
- `plotly.graph_objects`: lower-level API for detailed control.

Most analysis workflows start with Plotly Express and move to Graph Objects when more control is needed.

In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Use a clean default template for charts.
px.defaults.template = "plotly_white"

import plotly
plotly.__version__

'6.7.0'

## 2. Sample Dataset

This notebook uses small local DataFrames so it can run without downloading datasets.

Plotly works very well with tidy data: each variable is a column and each observation is a row.

In [3]:
sales = pd.DataFrame({
    "month": ["Jan", "Jan", "Jan", "Feb", "Feb", "Feb", "Mar", "Mar", "Mar", "Apr", "Apr", "Apr", "May", "May", "May", "Jun", "Jun", "Jun"],
    "region": ["North", "South", "West"] * 6,
    "channel": ["Online", "Retail", "Online", "Retail", "Online", "Retail", "Online", "Retail", "Online", "Retail", "Online", "Retail", "Online", "Retail", "Online", "Retail", "Online", "Retail"],
    "revenue": [12000, 9800, 10500, 14500, 13200, 11800, 16000, 14200, 12500, 18500, 15800, 15000, 21000, 17500, 16800, 24000, 19800, 18400],
    "profit": [3200, 2100, 2500, 3900, 3500, 2700, 4300, 3700, 2900, 5100, 4200, 3800, 6100, 4700, 4300, 7200, 5300, 4900],
    "orders": [120, 90, 100, 138, 125, 112, 150, 136, 118, 170, 148, 140, 190, 160, 152, 220, 178, 166],
    "customer_rating": [4.2, 3.9, 4.0, 4.3, 4.1, 4.0, 4.5, 4.2, 4.1, 4.6, 4.3, 4.2, 4.7, 4.4, 4.3, 4.8, 4.5, 4.4]
})

sales["profit_margin"] = (sales["profit"] / sales["revenue"] * 100).round(2)
sales["avg_order_value"] = (sales["revenue"] / sales["orders"]).round(2)

sales.head()

,month,region,channel,revenue,profit,orders,customer_rating,profit_margin,avg_order_value
0,Jan,North,Online,12000,3200,120,4.2,26.67,100.00
1,Jan,South,Retail,9800,2100,90,3.9,21.43,108.89
2,Jan,West,Online,10500,2500,100,4.0,23.81,105.00
3,Feb,North,Retail,14500,3900,138,4.3,26.90,105.07
4,Feb,South,Online,13200,3500,125,4.1,26.52,105.60


## 3. Plotly Express Line Chart

`px.line()` is useful for trends over time or ordered categories.

Useful arguments:

- `color`: split lines by category.
- `markers=True`: show data points.
- `hover_data`: control details shown on hover.

In [4]:
fig = px.line(
    sales,
    x="month",
    y="revenue",
    color="region",
    markers=True,
    hover_data=["orders", "profit_margin"],
    title="Revenue Trend by Region"
)

fig.update_layout(xaxis_title="Month", yaxis_title="Revenue")
fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## 4. Bar Chart

`px.bar()` compares categories.

Use `barmode="group"` for side-by-side bars and `barmode="stack"` for stacked bars.

In [ ]:
region_summary = sales.groupby(["region", "channel"], as_index=False)["revenue"].sum()

fig = px.bar(
    region_summary,
    x="region",
    y="revenue",
    color="channel",
    barmode="group",
    text_auto=True,
    title="Total Revenue by Region and Channel"
)

fig.update_layout(xaxis_title="Region", yaxis_title="Total Revenue")
fig.show()

## 5. Scatter Plot

`px.scatter()` shows relationships between numeric variables.

Plotly makes scatter plots especially useful because hover labels show exact values.

In [ ]:
fig = px.scatter(
    sales,
    x="orders",
    y="revenue",
    color="region",
    symbol="channel",
    size="profit_margin",
    hover_name="month",
    title="Orders vs Revenue"
)

fig.update_traces(marker=dict(line=dict(width=1, color="DarkSlateGrey")))
fig.show()

## 6. Bubble Chart

A bubble chart is a scatter plot where marker size encodes another numeric variable.

Use size carefully: bubbles are good for broad comparison, not exact reading.

In [ ]:
fig = px.scatter(
    sales,
    x="avg_order_value",
    y="profit_margin",
    size="revenue",
    color="region",
    hover_name="month",
    size_max=45,
    title="Profit Margin vs Average Order Value"
)

fig.update_layout(xaxis_title="Average Order Value", yaxis_title="Profit Margin %")
fig.show()

## 7. Pie and Donut Charts

`px.pie()` shows part-to-whole relationships.

A donut chart is a pie chart with a hole in the middle using `hole`.

In [ ]:
channel_summary = sales.groupby("channel", as_index=False)["revenue"].sum()

fig = px.pie(
    channel_summary,
    names="channel",
    values="revenue",
    hole=0.45,
    title="Revenue Share by Channel"
)

fig.update_traces(textposition="inside", textinfo="percent+label")
fig.show()

## 8. Histogram

`px.histogram()` shows the distribution of a numeric variable.

It can also compare distributions using `color`.

In [ ]:
fig = px.histogram(
    sales,
    x="revenue",
    color="channel",
    nbins=8,
    marginal="box",
    title="Revenue Distribution by Channel"
)

fig.update_layout(xaxis_title="Revenue", yaxis_title="Count")
fig.show()

## 9. Box Plot

`px.box()` shows median, quartiles, spread, and potential outliers.

It is useful for comparing distributions across categories.

In [ ]:
fig = px.box(
    sales,
    x="region",
    y="profit_margin",
    color="channel",
    points="all",
    title="Profit Margin Distribution by Region"
)

fig.update_layout(yaxis_title="Profit Margin %")
fig.show()

## 10. Violin Plot

`px.violin()` shows distribution shape.

It can show box plot details and individual points at the same time.

In [5]:
fig = px.violin(
    sales,
    x="channel",
    y="avg_order_value",
    color="channel",
    box=True,
    points="all",
    title="Average Order Value Distribution"
)

fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## 11. Heatmap

`px.imshow()` is useful for matrix-style charts.

Heatmaps are good for pivot tables, correlations, confusion matrices, and intensity maps.

In [ ]:
revenue_matrix = sales.pivot_table(
    index="region",
    columns="month",
    values="revenue",
    aggfunc="sum"
)

fig = px.imshow(
    revenue_matrix,
    text_auto=True,
    color_continuous_scale="Blues",
    title="Revenue Heatmap by Region and Month"
)

fig.update_layout(xaxis_title="Month", yaxis_title="Region")
fig.show()

## 12. Correlation Heatmap

Correlation heatmaps summarize relationships between numeric columns.

Values close to `1` mean strong positive correlation; values close to `-1` mean strong negative correlation.

In [ ]:
numeric_columns = ["revenue", "profit", "orders", "customer_rating", "profit_margin", "avg_order_value"]
corr = sales[numeric_columns].corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Correlation Heatmap"
)

fig.show()

## 13. Area Chart

`px.area()` shows cumulative contribution over an ordered axis.

Area charts work well for part-to-whole trend views.

In [ ]:
monthly_channel = sales.groupby(["month", "channel"], as_index=False)["revenue"].sum()

fig = px.area(
    monthly_channel,
    x="month",
    y="revenue",
    color="channel",
    title="Revenue Contribution by Channel"
)

fig.update_layout(yaxis_title="Revenue")
fig.show()

## 14. Treemap

`px.treemap()` shows hierarchical part-to-whole structure.

It is useful when you need to compare category sizes across multiple levels.

In [ ]:
fig = px.treemap(
    sales,
    path=["region", "channel", "month"],
    values="revenue",
    color="profit_margin",
    color_continuous_scale="Viridis",
    title="Revenue Treemap by Region, Channel, and Month"
)

fig.show()

## 15. Sunburst Chart

`px.sunburst()` is another way to visualize hierarchical data.

It is useful when the hierarchy itself is important to explore interactively.

In [ ]:
fig = px.sunburst(
    sales,
    path=["region", "channel"],
    values="profit",
    color="region",
    title="Profit Sunburst by Region and Channel"
)

fig.show()

## 16. Graph Objects

Graph Objects give lower-level control than Plotly Express.

Use Graph Objects when you want to manually define traces, combine chart types, or customize details deeply.

In [ ]:
monthly_total = sales.groupby("month", as_index=False).agg(revenue=("revenue", "sum"), profit=("profit", "sum"))

fig = go.Figure()

# Add a bar trace for revenue.
fig.add_trace(go.Bar(x=monthly_total["month"], y=monthly_total["revenue"], name="Revenue"))

# Add a line trace for profit.
fig.add_trace(go.Scatter(x=monthly_total["month"], y=monthly_total["profit"], name="Profit", mode="lines+markers"))

fig.update_layout(
    title="Monthly Revenue and Profit",
    xaxis_title="Month",
    yaxis_title="Amount",
    barmode="group"
)

fig.show()

## 17. Subplots

`make_subplots()` creates dashboard-style figures.

Each subplot receives traces with `row` and `col` arguments.

In [ ]:
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=("Revenue Trend", "Regional Revenue", "Orders vs Revenue", "Revenue Heatmap"),
    specs=[[{"type": "xy"}, {"type": "xy"}], [{"type": "xy"}, {"type": "heatmap"}]]
)

fig.add_trace(go.Scatter(x=monthly_total["month"], y=monthly_total["revenue"], mode="lines+markers", name="Revenue"), row=1, col=1)
fig.add_trace(go.Bar(x=region_summary["region"], y=region_summary["revenue"], name="Region Revenue"), row=1, col=2)
fig.add_trace(go.Scatter(x=sales["orders"], y=sales["revenue"], mode="markers", name="Orders vs Revenue"), row=2, col=1)
fig.add_trace(go.Heatmap(z=revenue_matrix.values, x=revenue_matrix.columns, y=revenue_matrix.index, colorscale="Blues", name="Heatmap"), row=2, col=2)

fig.update_layout(title="Interactive Sales Dashboard", height=750, showlegend=False)
fig.show()

## 18. Animation

Plotly Express can create animations using `animation_frame`.

Animations are useful for showing how relationships change over time.

In [ ]:
fig = px.scatter(
    sales,
    x="orders",
    y="revenue",
    size="profit",
    color="region",
    animation_frame="month",
    range_x=[80, 230],
    range_y=[9000, 25000],
    title="Animated Orders vs Revenue by Month"
)

fig.show()

## 19. 3D Scatter Plot

`px.scatter_3d()` creates interactive 3D plots.

3D charts can be rotated, zoomed, and explored in the notebook.

In [ ]:
fig = px.scatter_3d(
    sales,
    x="orders",
    y="revenue",
    z="profit_margin",
    color="region",
    symbol="channel",
    hover_name="month",
    title="3D View: Orders, Revenue, and Profit Margin"
)

fig.show()

## 20. Map Plot

`px.scatter_mapbox()` creates interactive map plots.

The `open-street-map` style does not require a Mapbox token.

In [ ]:
stores = pd.DataFrame({
    "city": ["Delhi", "Mumbai", "Bengaluru", "Hyderabad", "Chennai"],
    "lat": [28.6139, 19.0760, 12.9716, 17.3850, 13.0827],
    "lon": [77.2090, 72.8777, 77.5946, 78.4867, 80.2707],
    "revenue": [82000, 76000, 91000, 68000, 72000],
    "stores": [8, 7, 9, 6, 6]
})

fig = px.scatter_mapbox(
    stores,
    lat="lat",
    lon="lon",
    size="revenue",
    color="stores",
    hover_name="city",
    zoom=4,
    height=500,
    title="Store Revenue by City"
)

fig.update_layout(mapbox_style="open-street-map")
fig.show()

## 21. Hover Data and Custom Labels

Good hover labels make interactive charts much more useful.

Use `hover_data` to include, hide, or format values.

In [ ]:
fig = px.scatter(
    sales,
    x="orders",
    y="revenue",
    color="region",
    hover_name="month",
    hover_data={
        "orders": True,
        "revenue": ":,.0f",
        "profit_margin": ":.2f",
        "avg_order_value": ":.2f"
    },
    labels={
        "orders": "Orders",
        "revenue": "Revenue",
        "profit_margin": "Profit Margin %",
        "avg_order_value": "Average Order Value"
    },
    title="Custom Hover Labels"
)

fig.show()

## 22. Templates and Color Palettes

Templates control the overall visual style.

Common templates include:

- `plotly`
- `plotly_white`
- `plotly_dark`
- `ggplot2`
- `seaborn`
- `simple_white`

In [ ]:
fig = px.bar(
    monthly_total,
    x="month",
    y="revenue",
    color="profit",
    color_continuous_scale="Viridis",
    template="plotly_dark",
    title="Revenue with Dark Template"
)

fig.show()

## 23. Updating Layout and Traces

Two common methods:

- `update_layout()` changes figure-level settings.
- `update_traces()` changes data trace styling.

This is how you polish a Plotly chart after creating it.

In [ ]:
fig = px.line(monthly_total, x="month", y="revenue", markers=True, title="Polished Revenue Trend")

fig.update_traces(line=dict(width=4), marker=dict(size=10))
fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Revenue",
    title_x=0.5,
    hovermode="x unified"
)

fig.show()

## 24. Exporting Plotly Figures

Plotly figures can be exported as interactive HTML files.

Examples:

```python
fig.write_html("chart.html")
fig.write_image("chart.png")
```

`write_image()` may require the `kaleido` package. HTML export works well for sharing interactive charts.

In [ ]:
fig = px.line(monthly_total, x="month", y="revenue", markers=True, title="Export Example")

# Uncomment this line when you want to save an interactive HTML chart.
# fig.write_html("plotly_revenue_chart.html")

fig.show()

## 25. Real-World Mini Project: Interactive Sales Dashboard

This final section combines several Plotly skills:

- Multiple subplots.
- Bar, line, scatter, and heatmap traces.
- Clean dashboard layout.
- Interactive hover and zoom controls.

In [ ]:
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=("Monthly Revenue", "Revenue by Region", "Orders vs Revenue", "Region-Month Heatmap"),
    specs=[[{"type": "xy"}, {"type": "xy"}], [{"type": "xy"}, {"type": "heatmap"}]]
)

fig.add_trace(go.Scatter(x=monthly_total["month"], y=monthly_total["revenue"], mode="lines+markers", name="Monthly Revenue"), row=1, col=1)

region_total = sales.groupby("region", as_index=False)["revenue"].sum()
fig.add_trace(go.Bar(x=region_total["region"], y=region_total["revenue"], name="Region Revenue"), row=1, col=2)

fig.add_trace(go.Scatter(
    x=sales["orders"],
    y=sales["revenue"],
    mode="markers",
    marker=dict(size=sales["profit_margin"] * 1.5, color=sales["profit"], colorscale="Viridis", showscale=True),
    text=sales["region"] + " - " + sales["month"],
    name="Orders vs Revenue"
), row=2, col=1)

fig.add_trace(go.Heatmap(
    z=revenue_matrix.values,
    x=revenue_matrix.columns,
    y=revenue_matrix.index,
    colorscale="Blues",
    name="Revenue Heatmap"
), row=2, col=2)

fig.update_layout(
    title="Interactive Sales Dashboard",
    height=800,
    template="plotly_white",
    showlegend=False
)

fig.show()

## 26. Quick Revision Checklist

- Use `plotly.express` for fast charts with DataFrames.
- Use `graph_objects` for lower-level control.
- Use `color`, `symbol`, `size`, and `hover_data` to encode more information.
- Use `update_layout()` for titles, axes, templates, legends, and sizing.
- Use `update_traces()` to style chart marks.
- Use `make_subplots()` for dashboards.
- Use `px.imshow()` for heatmaps and matrix data.
- Use `animation_frame` for animated visualizations.
- Use `scatter_3d()` for interactive 3D exploration.
- Use `write_html()` to share interactive charts.